# NLP Assignment 5: UFO/UAP and Edwin Drood

This Colab-compatible notebook presents the completed UFO/UAP and Edwin Drood investigations, including methods, tables, visual evidence, conclusions, and limitations.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/Bershco/nlp_ufo_drood.git'
if not Path('nlp_ass5').exists():
    if not Path('nlp_ufo_drood/nlp_ass5').exists():
        !git clone {REPO_URL} nlp_ufo_drood
    %cd nlp_ufo_drood

!pip install -q -r requirements.txt

In [ ]:
import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
if not (ROOT / 'nlp_ass5').exists():
    raise FileNotFoundError('Repository setup failed; check that the GitHub repository is accessible.')

## Pipeline source and optional regeneration

The notebook is the submission interface; the complete executed pipeline is maintained in importable Python modules so the same code can run locally or in Colab without duplicating implementations. The relevant source files are:

- `nlp_ass5/ufo.py`: data loading, PDF-text integration, NER, exploration, semantic retrieval, detailed scoring, candidate export, manual-review support, figures, and report generation.
- `nlp_ass5/drood.py`: text preparation, character tables, contextual language, suspect scoring, embedding-assisted clue retrieval, scene clustering, Dickens comparison, figures, and report generation.
- `nlp_ass5/run_all.py`: the shared entry point that calls both complete pipelines.

The generated outputs are included in the repository, so regeneration is optional. The UFO run requires the manually acquired Kaggle and PURSUE source files and may take substantial time.

In [ ]:
# Complete pipeline entry points used to produce the submitted artifacts:
from nlp_ass5 import drood, ufo
from nlp_ass5.run_all import main as run_both_assignments

PIPELINE_SOURCES = {
    'UFO/UAP': ROOT / 'nlp_ass5/ufo.py',
    'Edwin Drood': ROOT / 'nlp_ass5/drood.py',
    'Combined runner': ROOT / 'nlp_ass5/run_all.py',
}
display(pd.DataFrame([{'pipeline': name, 'source_file': str(path), 'lines': len(path.read_text(encoding='utf-8').splitlines())} for name, path in PIPELINE_SOURCES.items()]))

# Uncomment exactly one command when full source data is available:
# ufo.run()             # Complete Assignment 1 pipeline
# drood.run()           # Complete Assignment 2 pipeline
# run_both_assignments()  # Both pipelines

# Assignment 1: UFO/UAP Sightings

The analysis combines 80,332 civilian Kaggle reports with 223 PURSUE rows. All 175 downloaded text-bearing official documents are represented: 138 metadata-attached rows, 61 standalone documents from Releases 1–3, and 24 remaining metadata-only records.

Transformer retrieval keeps up to 120 Kaggle candidates per usable PURSUE record. All 24,240 retrieved pairs receive detailed transformer, TF-IDF, lexical, date, location, and entity scoring. There is no minimum-score cutoff; only after global sorting are the top 500 exported. The complete implementation executed for these outputs is `nlp_ass5/ufo.py`, through `ufo.run()`.

In [ ]:
unified = pd.read_csv(ROOT / 'data/processed/ufo_unified.csv', low_memory=False)
reviews = pd.read_csv(ROOT / 'outputs/reports/ufo_manual_validation_completed.csv', keep_default_na=False)
display(unified.groupby(['source', 'text_kind']).size().rename('records').reset_index())
display(reviews[['candidate_rank', 'kaggle_record_id', 'pursue_record_id', 'manual_label', 'manual_notes']])

## Exploration and visualizations

Frequent terms and normalized shapes summarize the dominant language and reported forms. Temporal and geographic plots show the much larger scale of the civilian collection and its reporting patterns. The offline map is included so the submission remains readable without web access.

In [ ]:
for filename in ['ufo_top_terms.png', 'ufo_shapes.png', 'ufo_temporal_trends.png', 'ufo_geographic_map_offline.png']:
    display(Image(filename=str(ROOT / 'outputs/figures' / filename)))

## Manual validation

All top 20 pairs were reviewed. Nineteen were classified as **possibly the same event** and one as **probably not the same event**. Five representative cases are discussed in detail in `outputs/reports/ufo_report.md`. The recurring orange/red-orb descriptions explain why these pairs rank highly, but the released official metadata is often broad or absent and cannot support true verification.

## Conclusion

The system found several plausible thematic correspondences, especially reports involving orange or red orbs, but insufficient date, location, and distinctive-event evidence prevents confidently establishing a cross-source duplicate. This is a substantive result: the public PURSUE releases do not provide enough information to resolve the remaining uncertainty. Transformer similarity successfully retrieved comparable narratives, while secondary signals helped distinguish a shared sighting type from evidence of one historical event.

## Limitations

- PURSUE dates may represent incident, report, publication, review, or administrative dates.
- Many released locations are missing or intentionally broad; no better public information is available.
- Long historical documents can contain multiple incidents, and OCR quality varies.
- Generic terms such as *light*, *orb*, *orange*, and *red* recur across unrelated sightings.
- Semantic similarity identifies related descriptions, not proof of event identity.

In [ ]:
display(Markdown((ROOT / 'outputs/reports/ufo_report.md').read_text(encoding='utf-8')))

# Assignment 2: Edwin Drood

The novel was split into chapters, paragraphs, and sentences, with aliases for nine principal characters. The analysis covers mentions, chapter appearances, paragraph co-occurrence, contextual words, sentiment/themes, normalized suspect scoring, hybrid clue retrieval, scene clustering, and comparison with all six earlier Dickens novels listed in the assignment. The complete implementation executed for these outputs is `nlp_ass5/drood.py`, through `drood.run()`.

## Method and main result

The suspect model normalizes four components: motive (25%), opportunity (30%), suspicious language (30%), and semantic narrative relevance (15%). Neville ranks first in surface evidence because Dickens explicitly associates him with jealousy, threat, blood, and proximity. Jasper ranks second numerically, but the audited cross-scene chain includes opium and imagined strangulation, crypt access, obsessive love for Rosa, secrecy, and control of the post-disappearance narrative. This supports Jasper as the stronger literary suspect.

In [ ]:
suspects = pd.read_csv(ROOT / 'data/processed/drood_suspect_scores.csv')
clues = pd.read_csv(ROOT / 'data/processed/drood_important_clues.csv')
context_words = pd.read_csv(ROOT / 'data/processed/drood_character_context_words.csv')
display(suspects[['suspect', 'motive_score', 'opportunity_score', 'suspicious_language_score', 'narrative_relevance_score', 'suspicion_score']])
clue_display = clues[['chapter', 'quote', 'characters_involved', 'why_important', 'supports_suspect']].replace({chr(8212): ', '}, regex=True)
display(clue_display)
display(context_words.groupby('character').head(10))

## Visual evidence

In [ ]:
for filename in ['drood_character_mentions.png', 'drood_mentions_by_chapter.png', 'drood_cooccurrence_network.png', 'drood_suspect_scores.png', 'drood_dickens_motifs.png']:
    display(Image(filename=str(ROOT / 'outputs/figures' / filename)))

## Conclusion

John Jasper is the strongest literary suspect with medium confidence. Neville is the strongest surface-language suspect and likely functions as deliberate misdirection. Jasper probably attempted to kill Edwin, but the missing body, Datchery's unexplained identity, and Dickens's repeated use of disguise and delayed revelation leave survival plausible. NLP exposes associations, progression, and narrative structure; it cannot recover Dickens's unwritten ending.

In [ ]:
display(Markdown((ROOT / 'outputs/reports/drood_report.md').read_text(encoding='utf-8')))